In [1]:
!pip install datasets transformers scikit-learn

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DefaultDataCollator
import torch
import numpy as np
from sklearn.metrics import f1_score

In [3]:
dataset = load_dataset("go_emotions")

num_labels = 28

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

In [5]:
def to_multi_hot(example):
    multi_hot = np.zeros(num_labels, dtype=np.float32)
    multi_hot[example["labels"]] = 1.0
    example["labels"] = multi_hot
    return example

dataset = dataset.map(to_multi_hot)

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

In [6]:
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [7]:
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["validation"].select(range(500))

print(len(train_dataset), len(val_dataset))

2000 500


In [8]:
print(dataset["train"][0]["labels"])
print(type(dataset["train"][0]["labels"]))

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1])
<class 'torch.Tensor'>


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
label_names = dataset["train"].features["labels"].feature.names

for i, name in enumerate(label_names):
    print(i, name)

0 admiration
1 amusement
2 anger
3 annoyance
4 approval
5 caring
6 confusion
7 curiosity
8 desire
9 disappointment
10 disapproval
11 disgust
12 embarrassment
13 excitement
14 fear
15 gratitude
16 grief
17 joy
18 love
19 nervousness
20 optimism
21 pride
22 realization
23 relief
24 remorse
25 sadness
26 surprise
27 neutral


In [11]:
tail_labels = [16, 21, 23, 19, 12, 24, 14, 8]

In [12]:
def tail_recall(labels, preds):
    recalls = []
    for i in tail_labels:
        true = labels[:, i]
        pred = preds[:, i]

        tp = ((true == 1) & (pred == 1)).sum()
        fn = ((true == 1) & (pred == 0)).sum()

        if tp + fn > 0:
            recalls.append(tp / (tp + fn))

    return np.mean(recalls) if recalls else 0


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    tail = tail_recall(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "tail_recall": tail,
    }

In [13]:
data_collator = DefaultDataCollator(return_tensors="pt")

class MultiLabelCollator:
    def __call__(self, features):
        batch = data_collator(features)
        batch["labels"] = batch["labels"].float()
        return batch

collator = MultiLabelCollator()

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to="none",
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collator,
)

In [16]:
trainer.train()

Step,Training Loss
50,0.353379
100,0.164446
150,0.151468
200,0.148451
250,0.153222
300,0.147092
350,0.141995


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.17785384941101073, metrics={'train_runtime': 81.9698, 'train_samples_per_second': 73.198, 'train_steps_per_second': 4.575, 'total_flos': 198793230336000.0, 'train_loss': 0.17785384941101073, 'epoch': 3.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.14867551624774933,
 'eval_micro_f1': 0.0,
 'eval_macro_f1': 0.0,
 'eval_tail_recall': 0.0,
 'eval_runtime': 1.932,
 'eval_samples_per_second': 258.796,
 'eval_steps_per_second': 16.563,
 'epoch': 3.0}

In [18]:
pred_output = trainer.predict(val_dataset)

logits = pred_output.predictions
labels = pred_output.label_ids

probs = 1 / (1 + np.exp(-logits))

print(probs[:5])
print("max prob:", probs.max())
print("mean prob:", probs.mean())

[[0.14705963 0.06486995 0.05838929 0.0641134  0.08836211 0.03338091
  0.04183966 0.06708987 0.02305226 0.04405038 0.05278444 0.0278731
  0.01423799 0.02549727 0.02130961 0.11761342 0.01071496 0.05598158
  0.09412014 0.01154122 0.0529637  0.0129176  0.0302212  0.01315147
  0.02156548 0.04741138 0.04561887 0.18102118]
 [0.07127053 0.05017931 0.03770021 0.05457816 0.06693826 0.02380307
  0.02731402 0.04090585 0.01491453 0.0360879  0.05149345 0.01624937
  0.00753776 0.01333971 0.01331045 0.03717773 0.00500333 0.02945891
  0.03946876 0.00734395 0.03622132 0.00635177 0.01941854 0.00743023
  0.01254207 0.024339   0.02802014 0.3897177 ]
 [0.08861692 0.04954606 0.03919316 0.05232945 0.07014367 0.02300034
  0.02564951 0.03876995 0.01418057 0.0340264  0.04597936 0.01712112
  0.00761609 0.01491107 0.0129869  0.04724058 0.00536367 0.03345744
  0.05234858 0.00748568 0.03742795 0.00652545 0.01971172 0.00773558
  0.01337668 0.02729767 0.02886155 0.28337306]
 [0.06905336 0.04745315 0.03711506 0.0555504

In [19]:
from sklearn.metrics import f1_score

thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

for t in thresholds:
    preds = (probs > t).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    #tail = tail_recall(labels, preds, tail_labels)
    tail = tail_recall(labels, preds)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.05
  Micro-F1: 0.1908
  Macro-F1: 0.0789
  Tail Recall: 0.0000

Threshold: 0.1
  Micro-F1: 0.3016
  Macro-F1: 0.0525
  Tail Recall: 0.0000

Threshold: 0.15
  Micro-F1: 0.3302
  Macro-F1: 0.0558
  Tail Recall: 0.0000

Threshold: 0.2
  Micro-F1: 0.3240
  Macro-F1: 0.0453
  Tail Recall: 0.0000

Threshold: 0.25
  Micro-F1: 0.3057
  Macro-F1: 0.0293
  Tail Recall: 0.0000

Threshold: 0.3
  Micro-F1: 0.2834
  Macro-F1: 0.0212
  Tail Recall: 0.0000



In [20]:
fine_thresholds = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]

print("=== Fine-grained threshold sweep ===\n")

for t in fine_thresholds:
    preds = (probs > t).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    #tail = tail_recall(labels, preds, tail_labels)
    tail = tail_recall(labels, preds)

    print(f"Threshold: {t:.2f}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

=== Fine-grained threshold sweep ===

Threshold: 0.05
  Micro-F1: 0.1908
  Macro-F1: 0.0789
  Tail Recall: 0.0000

Threshold: 0.06
  Micro-F1: 0.2061
  Macro-F1: 0.0682
  Tail Recall: 0.0000

Threshold: 0.07
  Micro-F1: 0.2358
  Macro-F1: 0.0575
  Tail Recall: 0.0000

Threshold: 0.08
  Micro-F1: 0.2686
  Macro-F1: 0.0529
  Tail Recall: 0.0000

Threshold: 0.09
  Micro-F1: 0.2877
  Macro-F1: 0.0544
  Tail Recall: 0.0000

Threshold: 0.10
  Micro-F1: 0.3016
  Macro-F1: 0.0525
  Tail Recall: 0.0000



In [21]:
# Define per-class thresholds

default_threshold = 0.1
tail_threshold = 0.05

threshold_vector = np.full(num_labels, default_threshold)
for i in tail_labels:
    threshold_vector[i] = tail_threshold

print("Threshold vector:")
print(threshold_vector)

Threshold vector:
[0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.05 0.1  0.1  0.1  0.05 0.1
 0.05 0.1  0.05 0.1  0.1  0.05 0.1  0.05 0.1  0.05 0.05 0.1  0.1  0.1 ]


In [22]:
# Apply per-class thresholds

preds = (probs > threshold_vector).astype(int)

micro = f1_score(labels, preds, average="micro", zero_division=0)
macro = f1_score(labels, preds, average="macro", zero_division=0)
#tail = tail_recall(labels, preds, tail_labels)
tail = tail_recall(labels, preds)

print("=== Per-class threshold results ===")
print(f"Micro-F1: {micro:.4f}")
print(f"Macro-F1: {macro:.4f}")
print(f"Tail Recall: {tail:.4f}")

=== Per-class threshold results ===
Micro-F1: 0.3014
Macro-F1: 0.0525
Tail Recall: 0.0000


In [23]:
# ==========================================
# EXPERIMENT 10 – CLASS-WEIGHTED LOSS
# ==========================================

In [24]:
# Make sure we use the larger subset setting here
train_dataset = dataset["train"].select(range(5000))
val_dataset = dataset["validation"].select(range(500))

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 5000
Validation size: 500


In [25]:
# Build label matrix from the current training subset
train_label_matrix = np.stack([train_dataset[i]["labels"].numpy() for i in range(len(train_dataset))])

label_counts = train_label_matrix.sum(axis=0)
num_samples = train_label_matrix.shape[0]

print("Label counts:")
for i, count in enumerate(label_counts):
    print(i, label_names[i], int(count))

# BCEWithLogitsLoss pos_weight
# Higher weight for rarer positive labels
pos_weight = (num_samples - label_counts) / (label_counts + 1e-6)
pos_weight = torch.tensor(pos_weight, dtype=torch.float)

print("\nPos weights:")
print(pos_weight)

Label counts:
0 admiration 506
1 amusement 269
2 anger 184
3 annoyance 260
4 approval 316
5 caring 119
6 confusion 148
7 curiosity 221
8 desire 78
9 disappointment 163
10 disapproval 255
11 disgust 90
12 embarrassment 28
13 excitement 100
14 fear 67
15 gratitude 319
16 grief 12
17 joy 187
18 love 253
19 nervousness 23
20 optimism 182
21 pride 9
22 realization 120
23 relief 18
24 remorse 69
25 sadness 162
26 surprise 125
27 neutral 1646

Pos weights:
tensor([  8.8814,  17.5874,  26.1739,  18.2308,  14.8228,  41.0168,  32.7838,
         21.6244,  63.1026,  29.6748,  18.6078,  54.5556, 177.5714,  49.0000,
         73.6269,  14.6740, 415.6666,  25.7380,  18.7628, 216.3913,  26.4725,
        554.5555,  40.6667, 276.7778,  71.4638,  29.8642,  39.0000,   2.0377])


In [26]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, pos_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [27]:
model_w = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

training_args_w = TrainingArguments(
    output_dir="./results_weighted",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to="none",
)

trainer_w = WeightedTrainer(
    model=model_w,
    args=training_args_w,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collator,
    pos_weight=pos_weight,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
trainer_w.train()
trainer_w.evaluate()

Step,Training Loss
50,1.296890
100,1.307823
150,1.182469
200,1.040269
250,1.036299
300,1.042559
350,0.912488
400,0.856767
450,0.753103
500,0.739258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.9538329839706421,
 'eval_micro_f1': 0.3718553459119497,
 'eval_macro_f1': 0.345485511444103,
 'eval_tail_recall': 0.715625,
 'eval_runtime': 1.9278,
 'eval_samples_per_second': 259.369,
 'eval_steps_per_second': 16.6,
 'epoch': 3.0}

In [29]:
pred_output_w = trainer_w.predict(val_dataset)

logits_w = pred_output_w.predictions
labels_w = pred_output_w.label_ids

probs_w = 1 / (1 + np.exp(-logits_w))

print(probs_w[:5])
print("max prob:", probs_w.max())
print("mean prob:", probs_w.mean())

[[0.20685008 0.2218464  0.08834683 0.22203597 0.31277657 0.06579778
  0.8986927  0.93476427 0.14196354 0.1597507  0.13293691 0.08080729
  0.13543932 0.4563992  0.06888603 0.08404869 0.06935183 0.13167045
  0.13282192 0.06154447 0.20960645 0.03394001 0.507688   0.07325625
  0.11038272 0.09381671 0.77698046 0.40627223]
 [0.12638366 0.04972337 0.2774459  0.49022943 0.48761195 0.81129366
  0.09734685 0.07658961 0.0808943  0.1122408  0.44918707 0.16244097
  0.02659285 0.03406636 0.03856082 0.06410278 0.04657079 0.03297386
  0.03365811 0.01482286 0.27497876 0.0138537  0.19589296 0.0591197
  0.04784844 0.08029135 0.02672701 0.77356094]
 [0.22043662 0.3596732  0.16958113 0.2579995  0.23030816 0.509356
  0.2134127  0.12598106 0.42456794 0.7578059  0.37377617 0.29385474
  0.2825815  0.21636857 0.27819306 0.23638141 0.67571694 0.34125975
  0.36542058 0.20591491 0.4442393  0.35099453 0.56262904 0.42532814
  0.8269572  0.9481551  0.20182204 0.21378326]
 [0.05341592 0.19345807 0.38388777 0.73801893 

In [30]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

for t in thresholds:
    preds_w = (probs_w > t).astype(int)

    micro = f1_score(labels_w, preds_w, average="micro", zero_division=0)
    macro = f1_score(labels_w, preds_w, average="macro", zero_division=0)
    tail = tail_recall(labels_w, preds_w)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.05
  Micro-F1: 0.1014
  Macro-F1: 0.0884
  Tail Recall: 0.9688

Threshold: 0.1
  Micro-F1: 0.1329
  Macro-F1: 0.1123
  Tail Recall: 0.9281

Threshold: 0.15
  Micro-F1: 0.1637
  Macro-F1: 0.1402
  Tail Recall: 0.8969

Threshold: 0.2
  Micro-F1: 0.1923
  Macro-F1: 0.1664
  Tail Recall: 0.7469

Threshold: 0.25
  Micro-F1: 0.2224
  Macro-F1: 0.1975
  Tail Recall: 0.7469

Threshold: 0.3
  Micro-F1: 0.2555
  Macro-F1: 0.2341
  Tail Recall: 0.7469



In [31]:
high_thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]

for t in high_thresholds:
    preds_w = (probs_w > t).astype(int)

    micro = f1_score(labels_w, preds_w, average="micro", zero_division=0)
    macro = f1_score(labels_w, preds_w, average="macro", zero_division=0)
    tail = tail_recall(labels_w, preds_w)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.35
  Micro-F1: 0.2849
  Macro-F1: 0.2608
  Tail Recall: 0.7469

Threshold: 0.4
  Micro-F1: 0.3211
  Macro-F1: 0.3007
  Tail Recall: 0.7469

Threshold: 0.45
  Micro-F1: 0.3525
  Macro-F1: 0.3277
  Tail Recall: 0.7312

Threshold: 0.5
  Micro-F1: 0.3719
  Macro-F1: 0.3455
  Tail Recall: 0.7156

Threshold: 0.55
  Micro-F1: 0.3933
  Macro-F1: 0.3672
  Tail Recall: 0.7156

Threshold: 0.6
  Micro-F1: 0.4228
  Macro-F1: 0.3881
  Tail Recall: 0.5810

Threshold: 0.7
  Micro-F1: 0.4560
  Macro-F1: 0.4219
  Tail Recall: 0.5714



In [32]:
# ==========================================
# EXPERIMENT 12 – OVERSAMPLING + WEIGHTED LOSS
# ==========================================

In [33]:
from datasets import concatenate_datasets

# Use the same 5000-sample base subset
train_dataset = dataset["train"].select(range(5000))
val_dataset = dataset["validation"].select(range(500))

tail_train_indices = []

for i in range(len(train_dataset)):
    label_vector = train_dataset[i]["labels"].numpy()
    if any(label_vector[idx] == 1 for idx in tail_labels):
        tail_train_indices.append(i)

tail_subset = train_dataset.select(tail_train_indices)
oversampled_train_dataset = concatenate_datasets([train_dataset, tail_subset])

print("Original train size:", len(train_dataset))
print("Tail subset size:", len(tail_subset))
print("Oversampled train size:", len(oversampled_train_dataset))

Original train size: 5000
Tail subset size: 295
Oversampled train size: 5295


In [34]:
oversampled_label_matrix = np.stack(
    [oversampled_train_dataset[i]["labels"].numpy() for i in range(len(oversampled_train_dataset))]
)

label_counts_os = oversampled_label_matrix.sum(axis=0)
num_samples_os = oversampled_label_matrix.shape[0]

print("Oversampled label counts:")
for i, count in enumerate(label_counts_os):
    print(i, label_names[i], int(count))

pos_weight_os = (num_samples_os - label_counts_os) / (label_counts_os + 1e-6)
pos_weight_os = torch.tensor(pos_weight_os, dtype=torch.float)

print("\nOversampled pos weights:")
print(pos_weight_os)

Oversampled label counts:
0 admiration 517
1 amusement 273
2 anger 185
3 annoyance 267
4 approval 323
5 caring 123
6 confusion 150
7 curiosity 224
8 desire 156
9 disappointment 173
10 disapproval 262
11 disgust 94
12 embarrassment 56
13 excitement 101
14 fear 134
15 gratitude 328
16 grief 24
17 joy 191
18 love 258
19 nervousness 46
20 optimism 195
21 pride 18
22 realization 127
23 relief 36
24 remorse 138
25 sadness 179
26 surprise 127
27 neutral 1654

Oversampled pos weights:
tensor([  9.2418,  18.3956,  27.6216,  18.8315,  15.3932,  42.0488,  34.3000,
         22.6384,  32.9423,  29.6069,  19.2099,  55.3298,  93.5536,  51.4257,
         38.5149,  15.1433, 219.6250,  26.7225,  19.5233, 114.1087,  26.1538,
        293.1667,  40.6929, 146.0833,  37.3696,  28.5810,  40.6929,   2.2013])


In [35]:
model_ow = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

training_args_ow = TrainingArguments(
    output_dir="./results_oversampled_weighted",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to="none",
)

trainer_ow = WeightedTrainer(
    model=model_ow,
    args=training_args_ow,
    train_dataset=oversampled_train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collator,
    pos_weight=pos_weight_os,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
trainer_ow.train()
trainer_ow.evaluate()

Step,Training Loss
50,1.317592
100,1.256591
150,1.154211
200,1.056404
250,0.932421
300,0.893849
350,0.809685
400,0.711494
450,0.671205
500,0.672802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.780470073223114,
 'eval_micro_f1': 0.39241549876339654,
 'eval_macro_f1': 0.3686155711035132,
 'eval_tail_recall': 0.75625,
 'eval_runtime': 1.9831,
 'eval_samples_per_second': 252.136,
 'eval_steps_per_second': 16.137,
 'epoch': 3.0}

In [37]:
pred_output_ow = trainer_ow.predict(val_dataset)

logits_ow = pred_output_ow.predictions
labels_ow = pred_output_ow.label_ids

probs_ow = 1 / (1 + np.exp(-logits_ow))

print(probs_ow[:5])
print("max prob:", probs_ow.max())
print("mean prob:", probs_ow.mean())

[[0.2728775  0.14714445 0.0766698  0.18847542 0.33025107 0.0793049
  0.88511944 0.9422964  0.21209753 0.1159704  0.12767802 0.07188308
  0.08046205 0.36672413 0.06878885 0.08028141 0.06042246 0.10708338
  0.2365488  0.05500168 0.18019253 0.02869209 0.45585218 0.06274468
  0.07877618 0.09068374 0.7381621  0.44044667]
 [0.08755982 0.03644747 0.27169973 0.44233644 0.4664313  0.8071201
  0.11922204 0.0608518  0.08721718 0.13560241 0.43629268 0.16607565
  0.02802417 0.02752661 0.04739798 0.06920207 0.05974609 0.02892059
  0.02821396 0.02138865 0.26650044 0.0112143  0.2933245  0.04475109
  0.04820378 0.08784491 0.03465577 0.76118267]
 [0.12196928 0.24321975 0.13160937 0.28994602 0.20835651 0.43684277
  0.12208661 0.06534506 0.28084645 0.7649932  0.39274642 0.21200363
  0.15321103 0.10334089 0.19529517 0.15232871 0.84856147 0.20680523
  0.20863229 0.24929099 0.23724607 0.45649353 0.5741645  0.3206144
  0.9076754  0.960126   0.16001059 0.1860538 ]
 [0.04213072 0.09405649 0.32775432 0.69977355 

In [38]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for t in thresholds:
    preds_ow = (probs_ow > t).astype(int)

    micro = f1_score(labels_ow, preds_ow, average="micro", zero_division=0)
    macro = f1_score(labels_ow, preds_ow, average="macro", zero_division=0)
    tail = tail_recall(labels_ow, preds_ow)

    print(f"Threshold: {t}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

Threshold: 0.3
  Micro-F1: 0.2819
  Macro-F1: 0.2651
  Tail Recall: 0.7562

Threshold: 0.4
  Micro-F1: 0.3355
  Macro-F1: 0.3205
  Tail Recall: 0.7562

Threshold: 0.5
  Micro-F1: 0.3924
  Macro-F1: 0.3686
  Tail Recall: 0.7562

Threshold: 0.6
  Micro-F1: 0.4322
  Macro-F1: 0.3982
  Tail Recall: 0.7312

Threshold: 0.7
  Micro-F1: 0.4579
  Macro-F1: 0.4302
  Tail Recall: 0.7216



In [39]:
# ==========================================
# EXPERIMENT 13 – PER-CLASS THRESHOLDING ON
# OVERSAMPLING + WEIGHTED MODEL
# ==========================================

default_threshold = 0.7
tail_threshold = 0.5

threshold_vector_ow = np.full(num_labels, default_threshold)
for i in tail_labels:
    threshold_vector_ow[i] = tail_threshold

preds_ow_pc = (probs_ow > threshold_vector_ow).astype(int)

micro = f1_score(labels_ow, preds_ow_pc, average="micro", zero_division=0)
macro = f1_score(labels_ow, preds_ow_pc, average="macro", zero_division=0)
tail = tail_recall(labels_ow, preds_ow_pc)

print("=== Per-class threshold results (OW model) ===")
print(f"Micro-F1: {micro:.4f}")
print(f"Macro-F1: {macro:.4f}")
print(f"Tail Recall: {tail:.4f}")
print("Tail threshold:", tail_threshold)
print("Default threshold:", default_threshold)

=== Per-class threshold results (OW model) ===
Micro-F1: 0.4458
Macro-F1: 0.4087
Tail Recall: 0.7562
Tail threshold: 0.5
Default threshold: 0.7


In [40]:
settings = [
    (0.7, 0.5),
    (0.7, 0.6),
    (0.6, 0.5),
]

for default_threshold, tail_threshold in settings:
    threshold_vector_ow = np.full(num_labels, default_threshold)
    for i in tail_labels:
        threshold_vector_ow[i] = tail_threshold

    preds_ow_pc = (probs_ow > threshold_vector_ow).astype(int)

    micro = f1_score(labels_ow, preds_ow_pc, average="micro", zero_division=0)
    macro = f1_score(labels_ow, preds_ow_pc, average="macro", zero_division=0)
    tail = tail_recall(labels_ow, preds_ow_pc)

    print(f"default={default_threshold}, tail={tail_threshold}")
    print(f"  Micro-F1: {micro:.4f}")
    print(f"  Macro-F1: {macro:.4f}")
    print(f"  Tail Recall: {tail:.4f}")
    print()

default=0.7, tail=0.5
  Micro-F1: 0.4458
  Macro-F1: 0.4087
  Tail Recall: 0.7562

default=0.7, tail=0.6
  Micro-F1: 0.4518
  Macro-F1: 0.4163
  Tail Recall: 0.7312

default=0.6, tail=0.5
  Micro-F1: 0.4276
  Macro-F1: 0.3905
  Tail Recall: 0.7562

